# Day 02 — LLM APIs, Client Libraries & Local Models

**Learning focus:** Understand how applications communicate with frontier models and open-weight models.

## Today's Goals

- Understand LLM APIs, endpoints, SDKs, and models
- Learn the difference between native provider SDKs and OpenAI-compatible APIs
- Practice calling Gemini using its Python client library
- Understand `messages`, `contents`, `input`, and response extraction
- Understand closed/proprietary vs open-weight models
- Run open-weight models locally with Ollama
- Learn the basics of knowledge distillation
- Learn how to stay updated with changing LLM APIs


## 1. Core Concepts

### LLM API
An API allows an application to communicate with an AI model through a provider's service.

```text
Python Application
        ↓
Client Library / SDK
        ↓
API Endpoint
        ↓
Provider's AI Model
        ↓
Response
```

### API Endpoint
The endpoint is the URL where the HTTP request is sent.

### Client Library / SDK
A client library is a programming package that simplifies API communication.

Examples:

- OpenAI → `openai`
- Google Gemini → `google-genai`
- Anthropic → `anthropic`

### Important distinction

```text
Model ≠ API ≠ SDK

Model → GPT / Gemini / Claude / Llama
API   → Responses / Interactions / Messages / Generate Content
SDK   → openai / google-genai / anthropic
```

The SDK is a convenient wrapper around HTTP API calls; it does not contain the provider's model.


## 2. Frontier Models vs Open-Weight Models

### Closed / Proprietary Frontier Models

Examples include GPT, Gemini, and Claude.

The model is hosted by the provider and is normally accessed through an API.

```text
Your Application
      ↓
Provider API
      ↓
Provider Infrastructure
      ↓
Model
```

### Open-Weight Models

Examples include Llama, Qwen, Gemma, DeepSeek, Mistral, and Phi.

Weights are made available under their respective licenses, allowing compatible models to be run locally or on your own infrastructure.

```text
Your Application
      ↓
Ollama / Transformers / vLLM
      ↓
Downloaded Model
      ↓
Your Computer / Cloud
```

> Note: "open-weight" is more precise than automatically calling every such model "open-source".


## 3. OpenAI API and Chat Completions

A common chat-style request uses a `messages` list:

```python
messages = [
    {
        "role": "user",
        "content": "Tell me a fun fact"
    }
]
```

Historically, OpenAI's Chat Completions API became a very popular interface and many providers implemented compatible endpoints.

For new OpenAI applications, check the current official documentation because OpenAI has newer API interfaces as well.


In [ ]:
# Example messages structure

message = "Hello! This is my first message to an LLM."

messages = [
    {
        "role": "user",
        "content": message
    }
]

messages


## 4. Gemini Native Python Client

Google provides its own Python client library:

```python
from google import genai
```

The API key can be kept in `.env` instead of hard-coding it.

### `.env`

```text
GEMINI_API_KEY=your_key_here
```

### Python


In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

print("Gemini client created")


## 5. Gemini Generate Content API

For the Generate Content interface:

- `client.models.generate_content(...)`
- `contents` is used for the input
- `response.text` extracts the generated text

Do not mix this interface with the Interactions API syntax.


In [ ]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Explain what an LLM is in one sentence."
)

print(response.text)


### `messages` vs Gemini `contents`

An OpenAI-style message may look like:

```python
{"role": "user", "content": "What is AI?"}
```

For a simple Gemini request, the user text can be extracted:

```python
contents=messages[0]["content"]
```

For system instructions, Gemini's configuration can be used rather than blindly passing an OpenAI-style `messages` list.


In [ ]:
from google.genai import types

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=messages[1]["content"],
    config=types.GenerateContentConfig(
        system_instruction=messages[0]["content"]
    )
)

print(response.text)


## 6. Gemini Interactions API

Gemini also provides an Interactions API.

The basic pattern is:

```python
interaction = client.interactions.create(
    model="...",
    input="..."
)

print(interaction.output_text)
```

Notice the terminology:

```text
Generate Content API
    contents → response.text

Interactions API
    input → interaction.output_text
```

**Do not mix `input` and `contents`, or `text` and `output_text`, between these interfaces.**


In [ ]:
interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input="Explain how AI works in a few words"
)

print(interaction.output_text)


## 7. OpenAI-Compatible Endpoints

Some providers expose an OpenAI-compatible endpoint.

This means the **OpenAI Python SDK** can be used as a generic client while the request is sent to another provider's endpoint.

Conceptually:

```text
OpenAI Python SDK
       ↓
Provider's OpenAI-compatible endpoint
       ↓
Provider's model
```

Important:

> Using `from openai import OpenAI` does not automatically mean an OpenAI model is being used. The `base_url` determines where the request is sent.


In [ ]:
# Generic OpenAI-compatible pattern
#
# from openai import OpenAI
#
# client = OpenAI(
#     api_key=os.getenv("PROVIDER_API_KEY"),
#     base_url="https://provider.example/v1"
# )
#
# response = client.chat.completions.create(
#     model="provider-model",
#     messages=messages
# )
#
# print(response.choices[0].message.content)


## 8. Ollama — Local Open-Weight Models

Ollama provides a convenient way to run compatible open-weight models locally.

```text
Python Application
       ↓
Ollama
       ↓
Local Model
       ↓
Your Computer
```

Ollama also exposes an OpenAI-compatible API locally.

Default endpoint:

```text
http://localhost:11434
```

The OpenAI-compatible API base URL is commonly:

```text
http://localhost:11434/v1
```


In [ ]:
import requests

# Check whether Ollama is running
response = requests.get("http://localhost:11434")

print(response.text)


In [ ]:
# OpenAI-compatible Ollama client

from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

# Example after installing a local model:
#
# response = ollama.chat.completions.create(
#     model="qwen3:8b",
#     messages=[
#         {"role": "user", "content": "Tell me a fun fact"}
#     ]
# )
#
# print(response.choices[0].message.content)


### Ollama troubleshooting

If you run:

```text
ollama serve
```

and get:

```text
Only one usage of each socket address is normally permitted
```

it usually means Ollama is already running and port `11434` is already occupied by its server.

In that case, check with:

```bash
ollama list
```

and use the running Ollama instance rather than starting a second server.


## 9. Knowledge Distillation

**Knowledge distillation** transfers useful behavior from a larger **teacher** model to a smaller **student** model.

```text
Large Teacher Model
        ↓
Teacher-generated knowledge / examples
        ↓
Small Student Model
        ↓
Cheaper + faster model
```

### Why?

- Lower inference cost
- Faster responses
- Smaller compute requirements
- Useful for specialized production tasks

Distillation is different from ordinary fine-tuning, although the techniques can be combined.


## 10. Staying Updated With LLM APIs

LLM APIs change quickly.

### What to monitor

1. Official documentation
2. Model documentation
3. API reference
4. Release notes / changelog
5. Migration guides
6. SDK releases
7. Pricing and rate limits

### Golden rule

> **Official documentation is the source of truth. Don't blindly copy old tutorials.**

Before using an API:

```text
Provider
   ↓
Current Model
   ↓
Current API
   ↓
SDK Version
   ↓
Pricing & Limits
```

Examples of provider-specific API styles can change over time, so always verify the current recommended interface.


## 11. Today's Errors & Lessons

### Error 1 — OpenAI API quota

A `429 insufficient_quota / credit_balance_exhausted` error means the API account has no usable API credits. It is different from a Python syntax error.

### Error 2 — Gemini `input` vs `contents`

`generate_content()` uses `contents`, while the Interactions API uses `input`.

### Error 3 — Gemini response extraction

Use:

```python
response.text
```

for a Generate Content response.

For an Interaction:

```python
interaction.output_text
```

### Error 4 — Ollama port already in use

If port `11434` is already occupied, Ollama may already be running. Do not start another `ollama serve` process unnecessarily.


## 12. Key Takeaways

- An **API endpoint** is the URL where requests are sent.
- A **client library/SDK** simplifies calls to an API.
- The **SDK is not the model**.
- Providers can expose native APIs and OpenAI-compatible APIs.
- Gemini has its own `google-genai` SDK.
- OpenAI-compatible endpoints can be accessed using the `openai` Python library.
- Ollama allows compatible open-weight models to run locally.
- `generate_content()` and `interactions.create()` use different parameter and response names.
- API documentation changes frequently, so official docs and changelogs matter.
- Open-weight models can reduce API costs when run locally, but they require local compute and may be less capable than frontier models.


## 13. Tomorrow / Next Practice

- Compare OpenAI, Gemini, and Anthropic API interfaces
- Practice streaming responses
- Learn structured outputs
- Learn function/tool calling
- Continue building small LLM applications

---

**Day 02 completed — LLM APIs, SDKs, endpoints, Gemini, OpenAI-compatible APIs, and Ollama.**
